# NZZ ContextAI — Pipeline Runner

In [ ]:
import sys, importlib
sys.path.insert(0, "../src")
sys.path.insert(0, "../scripts")

import config, ingest, experiment, evaluate
importlib.reload(config)
importlib.reload(ingest)
importlib.reload(experiment)
importlib.reload(evaluate)

print("Imports OK")

## 1 — Indexierung

In [ ]:
ingest.ingest(month="2025_12", reset=True)

## 1 — Referenzantworten generieren

In [ ]:
import build_expected_answers
importlib.reload(build_expected_answers)

build_expected_answers.main()

## 2 — Ground Truth prüfen

In [ ]:
import json
from config import EVAL_GROUND_TRUTH

with open(EVAL_GROUND_TRUTH, encoding="utf-8") as f:
    samples = [json.loads(line) for line in f if line.strip()]

complete = [s for s in samples if len(s.get("expected_answer") or "") > 80]
print(f"Ground-Truth Einträge: {len(samples)}")
print(f"Mit Referenzantwort:   {len(complete)} / {len(samples)}")
print()
for s in samples:
    answer = s.get("expected_answer") or ""
    status = "✓" if len(answer) > 80 else "✗"
    print(f"  {status}  {s['query'][:65]}")

## 3 — Vollständige Pipeline + MLflow

In [ ]:
from experiment import run_experiment

run_experiment()

## 4 — MLflow UI starten

In [ ]:
import subprocess, sys

proc = subprocess.Popen(
    [sys.executable, "-m", "mlflow", "ui",
     "--backend-store-uri", "sqlite:///../mlflow.db",
     "--port", "5000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print("MLflow UI läuft → http://localhost:5000")
print(f"(Prozess-ID: {proc.pid} — Kernel neu starten zum Beenden)")

## 5 — Einzelne Query testen

In [ ]:
from embed import get_chroma_collection
from retrieval import load_models, retrieve
from generate import generate
from config import (
    CHROMA_PATH, CHROMA_COLLECTION,
    USE_RERANKING, EVAL_TOP_K_RETRIEVAL, EVAL_TOP_K_FINAL,
)

collection      = get_chroma_collection(CHROMA_PATH, CHROMA_COLLECTION)
model, reranker = load_models(use_reranking=USE_RERANKING)
print(f"Bereit — {collection.count():,} Chunks in ChromaDB")

In [ ]:
QUERY = "Was berichtete die NZZ über die SNB?"

chunks = retrieve(QUERY, model, collection, reranker,
                  top_k_retrieval=EVAL_TOP_K_RETRIEVAL,
                  top_k_rerank=EVAL_TOP_K_FINAL)

print(f"Top-{len(chunks)} Quellen:\n")
for i, c in enumerate(chunks, 1):
    score = c.get("rerank_score", c.get("similarity_score", 0))
    print(f"  [{i}] {c['title'][:65]}  (score: {score:.3f})")

print("\nAntwort:\n")
print(generate(QUERY, chunks))